# Chess Move Error Detection

## Problem

Each example is the first **40 plies** of a real chess game in SAN notation, where exactly
one of the first **10 plies** has been replaced by the move played at that same ply in a
*different* real game. The task is to recover the position of the substituted move, i.e. a
**10-class classification problem** over positions `1,...,10`. The moves after ply 10 are
part of the input: a corruption early in the opening leaves traces in what follows, so the
tail of the sequence carries signal about the head.

The dataset provides 400,000 training and 50,000 test examples, with three columns:
`sequence` (the 40 moves), `error_position` (the target, 1-10) and `correct_move` (the
original move — used here only for inspection, never as an input feature).

## Constraints

No external chess knowledge of any kind: no engines, opening books, move generators,
legality checks, or pretrained chess models. The rules of the game are treated as unknown,
and moves are handled as opaque string tokens throughout. The model must stay within
**6,000,000 trainable parameters**.

## Approach and stack

TensorFlow/Keras on Colab (GPU runtime), with experiment tracking on Weights & Biases.
The notebook runs top to bottom as a single pipeline: tokenizer → `tf.data` → model →
training → evaluation. The model below is a deliberately small bidirectional GRU baseline,
built first to validate the whole pipeline end to end; the architecture work builds on it
from there (see "Next steps").

## Setup

In [ ]:
!pip -q install gdown wandb

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CONFIG = {
    "seed": SEED,
    "seq_len": 40,
    "embed_dim": 64,
    "gru_units": 128,
    "dense_units": 128,
    "dropout": 0.2,
    "batch_size": 256,
    "epochs": 10,
    "learning_rate": 1e-3,
    "val_fraction": 0.1,
    "max_param_budget": 6_000_000,
}

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
CONFIG

## Data

In [ ]:
import gdown

TRAIN_FILE_ID = "1xyggntfZ2-6BTAagxJm-tFKDXerTAs8c"
TEST_FILE_ID  = "1VozRpr3dlVpA17BL-7ALCaOXa2vCe64J"

TRAIN_PATH = "chess_error_detection_train.csv"
TEST_PATH  = "chess_error_detection_test.csv"

if not os.path.exists(TRAIN_PATH):
    gdown.download(id=TRAIN_FILE_ID, output=TRAIN_PATH, quiet=False)

if not os.path.exists(TEST_PATH):
    gdown.download(id=TEST_FILE_ID, output=TEST_PATH, quiet=False)

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)

# Assumptions the rest of the pipeline relies on: fixed 40-move sequences and
# targets in 1..10. Checked once here so that any surprise fails loudly and early.
for name, df in [("train", train_df), ("test", test_df)]:
    lengths = df["sequence"].str.split().str.len()
    assert set(df.columns) == {"sequence", "error_position", "correct_move"}
    assert (lengths == CONFIG["seq_len"]).all(), f"{name}: variable-length sequences"
    assert df["error_position"].between(1, 10).all(), f"{name}: target out of range"

display(train_df.head())

### Class distribution

The corrupted position is sampled uniformly over the first 10 plies, so a random
classifier scores about **10%** Accuracy@1. That is the reference to beat.

In [ ]:
distribution = pd.DataFrame({
    "train": train_df["error_position"].value_counts(normalize=True).sort_index(),
    "test": test_df["error_position"].value_counts(normalize=True).sort_index(),
})

display(distribution)

### One example

Printing a single game with the corrupted ply marked makes the task concrete: nothing in
the sequence looks locally impossible, which is why the problem is hard without chess
knowledge.

In [ ]:
row = train_df.sample(1, random_state=SEED).iloc[0]
pos = int(row["error_position"])

for i, move in enumerate(row["sequence"].split(), start=1):
    marker = "  <-- corrupted" if i == pos else ""
    print(f"{i:2d}: {move}{marker}")

print("Correct move:", row["correct_move"])

## Evaluation metric

The official metric is **Accuracy@1**: a prediction counts as correct only when the
position with the highest predicted probability is exactly the corrupted one. The model
outputs a distribution over classes `0,...,9`, so `argmax` is shifted by one to match the
dataset convention `1,...,10`.

In [ ]:
def accuracy_at_1(y_true, probabilities):
    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)

    assert probabilities.ndim == 2
    assert probabilities.shape[1] == 10
    assert len(y_true) == len(probabilities)

    y_pred = np.argmax(probabilities, axis=1) + 1

    return np.mean(y_pred == y_true)

## Train / validation split

`test_df` is left untouched for the final evaluation. The validation set is carved out of
`train_df` only, stratified on `error_position` so all 10 classes stay balanced.

In [ ]:
train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=CONFIG["val_fraction"],
    random_state=SEED,
    stratify=train_df["error_position"],
)

print("Train split:", train_split_df.shape)
print("Val split:  ", val_split_df.shape)
print("Test (held out):", test_df.shape)

## Tokenizer

Moves are treated as opaque strings: the vocabulary is built **only from the training
split**, with no legality checks or move generators involved — just the tokens that happen
to occur in the data. `StringLookup` maps anything unseen to a single out-of-vocabulary
index, so the model still receives something for tokens it never trained on. The measured
OOV rate on validation is reported in the diagnostics section at the end.

In [ ]:
def build_vocab(sequences):
    tokens = set()
    for s in sequences:
        tokens.update(s.split())
    return sorted(tokens)


vocab = build_vocab(train_split_df["sequence"])
vocab_set = set(vocab)
print("Vocabulary size (train split only):", len(vocab))

lookup = tf.keras.layers.StringLookup(vocabulary=vocab, oov_token="[UNK]")
VOCAB_SIZE = lookup.vocabulary_size()
print("StringLookup size (incl. OOV):", VOCAB_SIZE)

## `tf.data` pipeline

Every sequence is exactly 40 moves (asserted above), so there is no padding to handle:
split on whitespace, look up ids, batch. The explicit `reshape` and `cast` pin down the
static shape and dtype that the model's `Input` declares, which `tf.strings.split` alone
would leave as `(None,)` / `int64`.

In [ ]:
def make_dataset(df, batch_size=CONFIG["batch_size"], shuffle=False):
    sequences = df["sequence"].values
    labels = (df["error_position"].values - 1).astype("int32")  # 0..9 for Keras

    ds = tf.data.Dataset.from_tensor_slices((sequences, labels))

    def encode(seq_str, label):
        tokens = tf.strings.split(seq_str)
        ids = lookup(tokens)
        ids = tf.reshape(ids, [CONFIG["seq_len"]])
        return tf.cast(ids, tf.int32), label

    ds = ds.map(encode, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(10_000, seed=SEED)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


# test_ds is deliberately not shuffled: the final evaluation aligns predictions
# with test_df["error_position"] positionally.
train_ds = make_dataset(train_split_df, shuffle=True)
val_ds = make_dataset(val_split_df)
test_ds = make_dataset(test_df)

## Baseline model

A bidirectional GRU reads the 40-move sequence and its final state is pooled into a 10-way
softmax over the candidate positions. Note what this architecture does *not* do: it
compresses the whole game into a single vector before scoring, so the 10 positions are
never compared against the context individually. That is the main structural limitation to
attack later.

No `mask_zero` on the embedding — index 0 is the OOV bucket, not padding, and the
sequences are fixed-length anyway.

In [ ]:
def build_model(
    vocab_size,
    seq_len=CONFIG["seq_len"],
    embed_dim=CONFIG["embed_dim"],
    gru_units=CONFIG["gru_units"],
    dense_units=CONFIG["dense_units"],
    dropout=CONFIG["dropout"],
    num_classes=10,
):
    inputs = tf.keras.Input(shape=(seq_len,), dtype=tf.int32, name="move_ids")
    x = tf.keras.layers.Embedding(vocab_size, embed_dim, name="move_embedding")(inputs)
    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.GRU(gru_units, return_sequences=False), name="bigru_encoder"
    )(x)
    x = tf.keras.layers.Dense(dense_units, activation="relu")(x)
    x = tf.keras.layers.Dropout(dropout)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="error_position")(x)
    return tf.keras.Model(inputs, outputs, name="chess_error_baseline")


model = build_model(VOCAB_SIZE)
model.summary()

trainable_params = int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))
print(f"Trainable parameters: {trainable_params:,} / {CONFIG['max_param_budget']:,}")
assert trainable_params <= CONFIG["max_param_budget"], "Model exceeds the 6M parameter budget!"

## Checkpointing

The Colab runtime is ephemeral: a timeout or a disconnection wipes everything under
`/content`, trained weights included. Checkpoints therefore go to Google Drive, keeping the
best epoch by `val_accuracy` across sessions. Set `USE_DRIVE = False` for short throwaway
runs, or if the Drive authorization flow fails — the fallback keeps the notebook runnable,
at the cost of losing the weights with the session.

In [ ]:
USE_DRIVE = True

CHECKPOINT_DIR = "/content/checkpoints"

if USE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        CHECKPOINT_DIR = "/content/drive/MyDrive/chess_error_detection/checkpoints"
    except Exception as err:
        print(f"Drive mount failed ({err}) - falling back to ephemeral runtime storage.")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "baseline_best.weights.h5")
print("Checkpoint path:", CHECKPOINT_PATH)

## Training

Metrics are logged to Weights & Biases. `wandb.login()` asks for an API key
(https://wandb.ai/authorize) once per session.

In [ ]:
import wandb
from wandb.integration.keras import WandbMetricsLogger

wandb.login()

run = wandb.init(
    project="chess-error-detection",
    config=CONFIG,
    name="baseline-bigru",
)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(CONFIG["learning_rate"]),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=CHECKPOINT_PATH,
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    save_weights_only=True,
    verbose=1,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG["epochs"],
    callbacks=[WandbMetricsLogger(), checkpoint_callback],
)

## Evaluation on the public test set

The best weights by `val_accuracy` are reloaded before evaluating, so the reported score
does not depend on whether the last epoch happened to be the best one.

In [ ]:
model.load_weights(CHECKPOINT_PATH)

test_probs = model.predict(test_ds)
test_labels = test_df["error_position"].values

test_accuracy = accuracy_at_1(test_labels, test_probs)
print(f"Test Accuracy@1: {test_accuracy:.4f}")

wandb.summary["trainable_params"] = trainable_params
wandb.log({"test_accuracy_at_1": test_accuracy})
wandb.finish()

## Diagnostics

Two checks that the aggregate accuracy hides. The **OOV rate** says whether whole-move
tokenization is losing information on rare moves. The **distribution of predictions** says
whether the model is spreading its answers over the 10 positions or collapsing onto a few
of them — a degenerate model can post a respectable accuracy while predicting almost the
same class every time.

In [ ]:
val_tokens = [t for s in val_split_df["sequence"] for t in s.split()]
oov_rate = sum(t not in vocab_set for t in val_tokens) / len(val_tokens)
print(f"OOV rate (validation): {oov_rate:.2e}")

pred = np.argmax(test_probs, axis=1) + 1
counts = np.bincount(pred, minlength=11)[1:]
expected = len(pred) / 10

print("\nPredictions per position (expected ~{:,.0f} each):".format(expected))
for position, count in enumerate(counts, start=1):
    print(f"{position:2d}: {count:6d}  ({count / expected:+.0%} vs uniform)")

## Results

| | |
|---|---|
| Architecture | Embedding(64) → BiGRU(128) → Dense(128, ReLU) → Dropout(0.2) → Dense(10, softmax) |
| Trainable parameters | see `trainable_params` above (budget: 6,000,000) |
| Test Accuracy@1 | see `test_accuracy` above |
| Random baseline | 0.10 |

## Next steps

- **Pointer / attention over positions.** The current model pools the whole game into one
  vector before the softmax, so each of the 10 candidate positions is never scored against
  the full context directly. This is the most promising change and the one the diagnostics
  point at.
- **Larger capacity.** The baseline uses a small fraction of the 6M budget, and the loss
  curves suggest underfitting rather than overfitting — there is room to grow width and
  depth before regularization becomes the binding concern.
- **Transformer encoder** in place of the BiGRU, with positional encodings.
- **Confusion matrix** over the 10 positions, to see whether errors concentrate on adjacent
  positions (blurred signal) or are spread uniformly (no signal).
- **Hyperparameter sweeps** via `wandb sweep` over learning rate, `embed_dim`, `gru_units`
  and dropout.
- Character-level tokenization is deliberately *not* a priority: the measured OOV rate is
  around 4e-05, so whole-move tokens already cover essentially the entire test vocabulary.